# F-K Dip Filtering Demo
Suppresses coherent linear noise (like ground-roll) using F-K dip filtering.


In [ ]:
%pip install bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall

## 1. Data Setup


In [ ]:
import segyio
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import row
from bokeh.models import LinearColorMapper, ColorBar
from seismicReader import readSeismicFile
from plotWiggles import plotWiggles
from pyseiskit import fk, gain, sourceData, normalizeTraces
from pyseiskit import PALETTES
output_notebook()

# Load specific shot gather (fldr=50)
filename = 'tac-204RL239.su'
gatherData, traceOffsets, timeSamples = readSeismicFile(filename, gatherKey=segyio.TraceField.FieldRecord, gatherIndex=49)

intervalTimeSamples = 0.004
if len(timeSamples) > 1:
    intervalTimeSamples = timeSamples[1] - timeSamples[0]
intervalSpaceSamples = 50.0  # dx=50m

# Apply TPOW to balance the visual energy
gatherData = gain.applyTimePowerGain(gatherData, 0.7, intervalTimeSamples)

# Apply trace-by-trace max normalization 
gatherData = normalizeTraces(gatherData, method='max')


## 2. Data Wiggle Plot


In [ ]:
show(plotWiggles(gatherData, timeSamples, traceOffsets, "Original Gather (before F-K)"))

## 3. Compute F-K Spectrum


In [ ]:
wavenumbers, frequencies, complexFKSpectrum = fk.computeFKSpectrum(gatherData, intervalTimeSamples, intervalSpaceSamples)
amplitudeFKSpectrum = np.abs(complexFKSpectrum)

def plotFkSpectrum(amplitudes, title, wavenumbers, frequencies):
    plotFigure = figure(
        width=500,
        height=600,
        title=title,
        x_axis_label="Wavenumber (cycles/m)",
        y_axis_label="Frequency (Hz)",
        x_range=(wavenumbers[0], wavenumbers[-1]), 
        y_range=(125, 0)
    ) # Zoomed to 125 Hz
    
    colorMapper = LinearColorMapper(palette=PALETTES['jet'])
    plotFigure.image(
        image=[amplitudes],
        x=wavenumbers[0],
        y=frequencies[0],
        dw=wavenumbers[-1]-wavenumbers[0],
        dh=frequencies[-1]-frequencies[0],
        color_mapper=colorMapper
    )

    plotFigure.add_layout(ColorBar(color_mapper=colorMapper, title="Amplitude"), 'right')
    return plotFigure

show(plotFkSpectrum(amplitudeFKSpectrum, "Spectrum before F-K filter", wavenumbers, frequencies))


## 4. Apply Dip Filter


In [ ]:
# Ground-roll rejection polygon
dipSlopes = [-0.0008, -0.0005, 0.0, 0.0005, 0.0008]
amplitudes = [0.0, 1.0, 1.0, 1.0, 0.0]

mutedFKSpectrum = fk.applyFKDipFilter(
    complexFKSpectrum,
	wavenumbers,
	frequencies,
    dipSlopes=dipSlopes,
    amplitudes=amplitudes
)

show(plotFkSpectrum(np.abs(mutedFKSpectrum), "Spectrum after F-K filter", wavenumbers, frequencies))


## 5. Apply Rectangular Mute (Surgical Mute)
Demonstrates muting a specific arbitrary rectangular region of the F-K spectrum.


In [ ]:
# Define an arbitrary rectangular box to surgically mute
minWavenumber = -0.01
maxWavenumber = -0.005
minFrequency = 20.0
maxFrequency = 30.0

surgicallyMutedFKSpectrum = fk.applyFKMute(
    complexFKSpectrum, 
    wavenumbers,
	frequencies,
    minWavenumber,
	maxWavenumber,
    minFrequency,
	maxFrequency,
    amplitudeMultiplier=0.4
)

show(plotFkSpectrum(np.abs(surgicallyMutedFKSpectrum), "Spectrum after Surgical Rectangular Mute", wavenumbers, frequencies))


## 6. Apply Tapered Mute (Seismic Unix style)
Demonstrates how to apply a smooth 2D surgical mute by providing coordinate nodes and amplitude arrays.


In [ ]:
# Reproducing the exact coordinates: [-0.01, -0.005] and [20.0, 30.0]
# The noise is in the top-left, so we create a sharp cut at the top (20Hz) and left (-0.01).
# Then we create a smooth fade towards the bottom (30Hz) and right (-0.005).
wavenumberNodes = [-0.01, -0.008, -0.005]
wavenumberAmplitudes  = [0,   0.6,    1.0]

frequencyNodes = [19.99, 20.0, 25.0, 30.0]
frequencyAmplitudes  = [1.0,   0,  0.5,  1.0]

taperedFKSpectrum = fk.applyFKTaperedMute(
    complexFKSpectrum, 
    wavenumbers,
	frequencies,
    wavenumberNodes,
    frequencyNodes,
    wavenumberAmplitudes,
    frequencyAmplitudes
)

show(plotFkSpectrum(np.abs(taperedFKSpectrum), "Spectrum after Tapered Rectangular Mute", wavenumbers, frequencies))


## 7. Inverse F-K Transform (Back to Time-Space)
Once the spectrum has been surgically mutated or dip-filtered, we perform an inverse 2D FFT to recover the seismic gather.

In [ ]:
# Determine original dimensions
numberSamples, numberTraces = gatherData.shape

# Apply Inverse F-K Transform for both hard and tapered mutes
hardMutedGather = fk.applyFKInverse(
    surgicallyMutedFKSpectrum, 
    numberSamples, 
    numberTraces
)

taperedGather = fk.applyFKInverse(
    taperedFKSpectrum, 
    numberSamples, 
    numberTraces
)

# Re-normalize after filtering for optimal visualization
hardMutedGather = normalizeTraces(hardMutedGather, method='max')
taperedGather = normalizeTraces(taperedGather, method='max')

# Plot the original, hard-muted, and tapered gathers side-by-side
show(
    row(
        plotWiggles(gatherData, timeSamples, traceOffsets, "Original Gather"),
        plotWiggles(hardMutedGather, timeSamples, traceOffsets, "Hard Mute (Gibbs Ringing)"),
        plotWiggles(taperedGather, timeSamples, traceOffsets, "Tapered Mute")
    )
)
